In [4]:
import requests
import pandas as pd
import time
import re
from datetime import datetime

# **Settings**

In [ ]:
# Paste your token here
TOKEN = ""
# Famous public repositories:
REPOS = [
    "tensorflow/tensorflow",
    "pytorch/pytorch",
    "microsoft/vscode",
    "facebook/react",
    "scikit-learn/scikit-learn",
    "kubernetes-client/python",
    "NousResearch/hermes-agent",
    "pingcap/tidb",
    "multica-ai/multica",
    "rustfs/rustfs",
    "django/django",
    "pallets/flask",
    "apache/airflow",
    "electron/electron",
    "ansible/ansible",
    "moby/moby",
    "langchain-ai/langchain",
    "Significant-Gravitas/AutoGPT",
    "OpenInterpreter/open-interpreter",
    "deepset-ai/haystack"
]

# Recommended:
# Start with 300 to test
# Then increase to 600 or 800 if your token/rate limit allows
COMMITS_PER_REPO = 500

BUG_WORDS = [
    "fix", "fixed", "fixes",
    "bug", "bugs",
    "error", "errors",
    "defect", "defects",
    "issue", "issues",
    "patch", "patched",
    "resolve", "resolved", "resolves",
    "hotfix"
]

HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"token {TOKEN}"
}

# **Helper Functions**

In [ ]:
def is_bug_fix_message(message):
    message = message.lower()
    for word in BUG_WORDS:
        if word in message:
            return 1
    return 0



def github_get(url, params=None):
    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=20
    )

    if response.status_code == 200:
        return response.json()

    elif response.status_code == 403:
        remaining = response.headers.get("X-RateLimit-Remaining")
        reset = response.headers.get("X-RateLimit-Reset")

        print("403 received")
        print("Remaining:", remaining)
        print("Response:", response.text)

        if remaining == "0" and reset:
            wait_seconds = int(reset) - int(time.time()) + 5

            if wait_seconds > 0:
                print(f"Actual rate limit hit. Waiting {wait_seconds} sec...")
                time.sleep(wait_seconds)

                retry = requests.get(
                    url,
                    headers=HEADERS,
                    params=params,
                    timeout=20
                )

                if retry.status_code == 200:
                    return retry.json()

        return None

    elif response.status_code == 404:
        return None

    else:
        print("Error:", response.status_code)
        print(response.text)
        return None

def get_commit_list(repo, total_commits):
    all_commits = []
    per_page = 100
    pages = (total_commits + per_page - 1) // per_page

    for page in range(1, pages + 1):
        url = f"https://api.github.com/repos/{repo}/commits"
        params = {"per_page": per_page, "page": page}
        data = github_get(url, params=params)

        if not data:
            break

        all_commits.extend(data)

        if len(data) < per_page:
            break

        time.sleep(0.2)

    return all_commits[:total_commits]


def get_commit_details(repo, sha):
    url = f"https://api.github.com/repos/{repo}/commits/{sha}"
    data = github_get(url)
    time.sleep(0.2)
    return data

# **Collect data**

In [ ]:
from datetime import datetime

all_rows = []

# tracks developer experience
developer_commit_count = {}

for repo in REPOS:
    print(f"\nCollecting from {repo} ...")

    commit_list = get_commit_list(repo, COMMITS_PER_REPO)
    repo_rows = []

    for i, item in enumerate(commit_list):
        sha = item["sha"]
        detail = get_commit_details(repo, sha)

        if detail is None:
            continue

        commit_info = detail.get("commit", {})
        author_info = commit_info.get("author", {}) or {}
        stats = detail.get("stats", {}) or {}
        parents = detail.get("parents", []) or []
        files = detail.get("files", []) or []

        message = commit_info.get("message", "") or ""
        author_name = author_info.get("name", "unknown")
        author_date = author_info.get("date", "")

        parent_sha = parents[0]["sha"] if len(parents) > 0 else None

        changed_files = []
        directories = set()

        for f in files:
            name = f.get("filename", "")

            if name:
                changed_files.append(name)

                if "/" in name:
                    directories.add(name.split("/")[0])

        # message length
        message_length = len(message)

        # commit hour/day
        try:
            dt = datetime.strptime(author_date, "%Y-%m-%dT%H:%M:%SZ")
            commit_hour = dt.hour
            commit_day = dt.weekday()
        except:
            commit_hour = -1
            commit_day = -1

        # developer experience
        if author_name not in developer_commit_count:
            developer_commit_count[author_name] = 0

        developer_experience = developer_commit_count[author_name]

        developer_commit_count[author_name] += 1

        row = {
            "repo": repo,
            "sha": sha,
            "parent_sha": parent_sha,
            "parent_count": len(parents),

            "author_name": author_name,
            "author_date": author_date,

            "message": message.replace("\n", " ").strip(),

            # software metrics
            "files_changed": len(changed_files),
            "additions": stats.get("additions", 0),
            "deletions": stats.get("deletions", 0),
            "total_changes": stats.get("total", 0),

            # engineered features
            "message_length": message_length,
            "commit_hour": commit_hour,
            "commit_day": commit_day,
            "developer_experience": developer_experience,
            "directories_changed": len(directories),

            "is_fix_commit": is_bug_fix_message(message)
        }

        repo_rows.append(row)

        if (i + 1) % 50 == 0:
            print(f"{repo}: {i+1} commits done")

    # parent labeling
    fix_parent_shas = set()

    for row in repo_rows:
        if row["is_fix_commit"] == 1 and row["parent_sha"] is not None:
            fix_parent_shas.add(row["parent_sha"])

    for row in repo_rows:
        row["buggy_label"] = 1 if row["sha"] in fix_parent_shas else 0

    all_rows.extend(repo_rows)

print("\nFinished collecting new raw data.")


tensorflow/tensorflow: 50 commits done
tensorflow/tensorflow: 100 commits done
tensorflow/tensorflow: 150 commits done
tensorflow/tensorflow: 200 commits done
tensorflow/tensorflow: 250 commits done
tensorflow/tensorflow: 300 commits done
tensorflow/tensorflow: 350 commits done
tensorflow/tensorflow: 400 commits done
tensorflow/tensorflow: 450 commits done
tensorflow/tensorflow: 500 commits done

pytorch/pytorch: 50 commits done
pytorch/pytorch: 100 commits done
pytorch/pytorch: 150 commits done
pytorch/pytorch: 200 commits done
pytorch/pytorch: 250 commits done
pytorch/pytorch: 300 commits done
pytorch/pytorch: 350 commits done
pytorch/pytorch: 400 commits done
pytorch/pytorch: 450 commits done
pytorch/pytorch: 500 commits done

microsoft/vscode: 50 commits done
microsoft/vscode: 100 commits done
microsoft/vscode: 150 commits done
microsoft/vscode: 200 commits done
microsoft/vscode: 250 commits done
microsoft/vscode: 300 commits done
microsoft/vscode: 350 commits done
microsoft/vscod

# **Save CSV**

In [ ]:
df_raw = pd.DataFrame(all_rows)
print(df_raw.shape)
df_raw.head()

(9999, 18)


,repo,sha,parent_sha,parent_count,author_name,author_date,message,files_changed,additions,deletions,total_changes,message_length,commit_hour,commit_day,developer_experience,directories_changed,is_fix_commit,buggy_label
0,tensorflow/tensorflow,06ba7ec486ef88d1e496dd0dfe5a8d0663de3343,62011cf34d82e6d168b976eff033b0b78d3afbe5,1,Kanish Anand,2026-05-12T14:55:37Z,"Disable `HloShardingV3` in ShardyXLA tests, ad...",1,4,0,4,98,14,1,0,1,0,0
1,tensorflow/tensorflow,62011cf34d82e6d168b976eff033b0b78d3afbe5,0181888c80f3c17e4beb2a9e9189e90d61be38c9,1,Eetu Sjöblom,2026-05-12T13:57:34Z,PR #41616: [ROCm] Add PCIe and interconnect ba...,11,554,13,567,638,13,1,0,1,0,0
2,tensorflow/tensorflow,0181888c80f3c17e4beb2a9e9189e90d61be38c9,33643e996d4b98d282be0044afdabb984ee6cda4,1,Andrei Ivanov,2026-05-12T13:55:53Z,PR #42425: Add SM103 HLO op profiles for B300 ...,2,946,0,946,938,13,1,0,1,0,0
3,tensorflow/tensorflow,33643e996d4b98d282be0044afdabb984ee6cda4,ad8b78590b697e2812d1d374971b2fffecbca201,1,Adrian Kuegel,2026-05-12T13:52:29Z,[XLA:GPU] Migrate more tests away from GpuCode...,7,166,207,373,420,13,1,0,1,1,0
4,tensorflow/tensorflow,ad8b78590b697e2812d1d374971b2fffecbca201,47ae76a1b7eab2fde3424728e01b1f7d623d787c,1,Ahmed Sagar,2026-05-12T13:45:44Z,PR #35762: Fix permutation sort expander dtype...,2,52,1,53,2442,13,1,0,1,1,1


In [ ]:
df_raw.to_csv("bug_prediction_raw_dataset_new.csv", index=False)
print("Saved: bug_prediction_raw_dataset_new.csv")

Saved: bug_prediction_raw_dataset_new.csv


In [5]:
df = pd.read_csv("bug_prediction_raw_dataset_new.csv")

In [6]:
df

,repo,sha,parent_sha,parent_count,author_name,author_date,message,files_changed,additions,deletions,total_changes,message_length,commit_hour,commit_day,developer_experience,directories_changed,is_fix_commit,buggy_label
0,tensorflow/tensorflow,06ba7ec486ef88d1e496dd0dfe5a8d0663de3343,62011cf34d82e6d168b976eff033b0b78d3afbe5,1,Kanish Anand,2026-05-12T14:55:37Z,"Disable `HloShardingV3` in ShardyXLA tests, ad...",1,4,0,4,98,14,1,0,1,0,0
1,tensorflow/tensorflow,62011cf34d82e6d168b976eff033b0b78d3afbe5,0181888c80f3c17e4beb2a9e9189e90d61be38c9,1,Eetu Sjöblom,2026-05-12T13:57:34Z,PR #41616: [ROCm] Add PCIe and interconnect ba...,11,554,13,567,638,13,1,0,1,0,0
2,tensorflow/tensorflow,0181888c80f3c17e4beb2a9e9189e90d61be38c9,33643e996d4b98d282be0044afdabb984ee6cda4,1,Andrei Ivanov,2026-05-12T13:55:53Z,PR #42425: Add SM103 HLO op profiles for B300 ...,2,946,0,946,938,13,1,0,1,0,0
3,tensorflow/tensorflow,33643e996d4b98d282be0044afdabb984ee6cda4,ad8b78590b697e2812d1d374971b2fffecbca201,1,Adrian Kuegel,2026-05-12T13:52:29Z,[XLA:GPU] Migrate more tests away from GpuCode...,7,166,207,373,420,13,1,0,1,1,0
4,tensorflow/tensorflow,ad8b78590b697e2812d1d374971b2fffecbca201,47ae76a1b7eab2fde3424728e01b1f7d623d787c,1,Ahmed Sagar,2026-05-12T13:45:44Z,PR #35762: Fix permutation sort expander dtype...,2,52,1,53,2442,13,1,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,deepset-ai/haystack,ac9455dc8df2fba03653aaa124ac47f3a1e2dfe1,5b8c5338c198eca9453af71fbe1ec32e1912f339,1,Stefano Fiorucci,2026-02-09T08:38:00Z,docs: add section on flexible connections for ...,4,147,79,226,183,8,0,71,1,1,1
9995,deepset-ai/haystack,5b8c5338c198eca9453af71fbe1ec32e1912f339,5f89e9c9e75b5476f423f3e17c782b6cd6a9a320,1,alex li,2026-02-09T08:28:29Z,docs: remove stale Marqo document store links ...,15,12,49,61,54,8,0,0,2,0,1
9996,deepset-ai/haystack,5f89e9c9e75b5476f423f3e17c782b6cd6a9a320,f581edcabc8a2fb0fe7f755257784bc575ea55f9,1,Stefano Fiorucci,2026-02-06T15:46:09Z,feat: Pipelines - support connection and conve...,7,519,152,671,549,15,4,72,3,1,0
9997,deepset-ai/haystack,f581edcabc8a2fb0fe7f755257784bc575ea55f9,079bee49f04a925d7ce660a508c24e2b7d224d3a,1,dependabot[bot],2026-02-06T14:27:00Z,chore(deps): bump fossas/fossa-action from 1.7...,1,1,1,2,614,14,4,370,1,0,1
